# 07. Model Comparison

## Objective

This notebook compares the retained logistic regression with a shallow decision tree. The experiment changes only the model family. Features, temporal folds, purging rules, validation observations, and the temporary 0.50 threshold remain identical. The locked test set is not evaluated.

## Work plan

1. Load the feature table and model-selection folds.
2. Rebuild the same purged temporal training sets.
3. Fit the reference logistic regression and a shallow decision tree.
4. Compare training and validation metrics on both folds.
5. Inspect the tree rules and feature importance.

## 1. Load modeling inputs

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    confusion_matrix, f1_score, precision_score, recall_score,
    roc_auc_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree

modeling_data = pd.read_parquet(
    "../data/processed/churn_snapshot_features.parquet"
)
model_selection_folds = pd.read_csv(
    "../data/interim/model_selection_folds.csv",
    parse_dates=["TrainStart", "ValidationStart", "ValidationEnd"],
)
model_selection_folds = model_selection_folds.loc[
    model_selection_folds["Approach"].eq("B")
].copy()

print("Modeling data:", modeling_data.shape)
print("Approach B folds:", len(model_selection_folds))

## 2. Fixed modeling contract

The experiment uses the four features retained in notebook 06. `WillChurnNext30Days` is the target and is never passed as a feature. Only development rows are used.

In [ ]:
feature_columns = [
    "RecencyDays",
    "PurchaseFrequency",
    "HasCancellation",
    "IsInChurnRiskWindow",
]
target_column = "WillChurnNext30Days"
development_data = modeling_data.loc[
    modeling_data["FinalSplit"].eq("development")
].copy()

print("Features:", feature_columns)
print("Development rows:", len(development_data))

## 3. Rebuild the purged temporal folds

A training snapshot is retained only when its complete 30-day label window ends no later than validation begins. This prevents training labels from using outcomes observed inside the validation period.

In [ ]:
def get_fold_data(fold):
    candidate_train = development_data["ReferenceDate"].between(
        fold["TrainStart"], fold["ValidationStart"], inclusive="left"
    )
    train_mask = (
        candidate_train
        & development_data["LabelEndDate"].le(fold["ValidationStart"])
    )
    validation_mask = development_data["ReferenceDate"].between(
        fold["ValidationStart"], fold["ValidationEnd"]
    )
    return (
        development_data.loc[train_mask].copy(),
        development_data.loc[validation_mask].copy(),
    )

fold_checks = []
for _, fold in model_selection_folds.iterrows():
    train, validation = get_fold_data(fold)
    fold_checks.append({
        "Fold": fold["Fold"],
        "TrainRows": len(train),
        "ValidationRows": len(validation),
        "LatestTrainLabelEnd": train["LabelEndDate"].max(),
        "ValidationStart": validation["ReferenceDate"].min(),
        "NoLabelOverlap": (
            train["LabelEndDate"].max()
            <= validation["ReferenceDate"].min()
        ),
    })
pd.DataFrame(fold_checks)

## 4. Models under comparison

The logistic regression is the reference selected in notebook 06. The decision tree is limited to a depth of 3 and at least 50 training observations per leaf. These restrictions make its rules readable and reduce overfitting. No class weighting or hyperparameter search is introduced yet, so the model family is the only substantive change.

In [ ]:
models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, solver="liblinear"),
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=3,
        min_samples_leaf=50,
        random_state=42,
    ),
}
models

## 5. Fit and evaluate both models

`PR_AUC` is the main ranking metric. Training metrics are shown only to diagnose overfitting. Model selection is based on validation performance and stability across both folds. The 0.50 threshold remains temporary.

In [ ]:
def classification_metrics(y_true, y_probability, threshold=0.50):
    y_pred = y_probability >= threshold
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[False, True]
    ).ravel()
    return {
        "ChurnRate": y_true.mean(),
        "ContactRate": y_pred.mean(),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "Specificity": tn / (tn + fp),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Accuracy": accuracy_score(y_true, y_pred),
        "BalancedAccuracy": balanced_accuracy_score(y_true, y_pred),
        "PR_AUC": average_precision_score(y_true, y_probability),
        "ROC_AUC": roc_auc_score(y_true, y_probability),
        "FalsePositives": fp,
        "FalseNegatives": fn,
    }

comparison_rows = []
fitted_models = {}
for _, fold in model_selection_folds.iterrows():
    train, validation = get_fold_data(fold)
    for model_name, model_template in models.items():
        model = clone(model_template)
        model.fit(
            train[feature_columns].astype(float), train[target_column]
        )
        fitted_models[(fold["Fold"], model_name)] = model

        for dataset_name, dataset in [
            ("train", train), ("validation", validation)
        ]:
            y_probability = model.predict_proba(
                dataset[feature_columns].astype(float)
            )[:, 1]
            comparison_rows.append({
                "Fold": fold["Fold"],
                "Model": model_name,
                "Dataset": dataset_name,
                "Rows": len(dataset),
                **classification_metrics(
                    dataset[target_column], y_probability
                ),
            })
comparison_results = pd.DataFrame(comparison_rows)

## 6. Read the results

The first table displays both training and validation performance. A large train-validation gap is evidence of overfitting. The second table reports the tree's validation change relative to logistic regression. A positive delta means the tree is higher on that metric.

In [ ]:
percentage_columns = [
    "ChurnRate", "ContactRate", "Precision", "Recall",
    "Specificity", "F1", "Accuracy",
    "BalancedAccuracy", "PR_AUC", "ROC_AUC",
]
comparison_display = comparison_results.copy()
comparison_display[percentage_columns] *= 100
display(comparison_display.round(2))

validation_results = comparison_results.loc[
    comparison_results["Dataset"].eq("validation")
].copy()
logistic_validation = validation_results.loc[
    validation_results["Model"].eq("Logistic Regression")
].set_index("Fold")
tree_validation = validation_results.loc[
    validation_results["Model"].eq("Decision Tree")
].set_index("Fold")
metric_columns = [
    "PR_AUC", "Precision", "Recall", "F1",
    "BalancedAccuracy", "FalsePositives", "FalseNegatives",
]
tree_delta = tree_validation[metric_columns] - logistic_validation[metric_columns]
tree_delta[[
    "PR_AUC", "Precision", "Recall", "F1", "BalancedAccuracy"
]] *= 100
tree_delta.add_prefix("TreeDelta_").round(2)

## 7. Inspect what the tree learned

Feature importance indicates which variables the tree used to reduce class impurity. It does not show whether a feature increases or decreases churn risk and should not be interpreted causally. The plotted rules provide the actual thresholds and interactions learned in each fold.

In [ ]:
importance_rows = []
for (fold_name, model_name), model in fitted_models.items():
    if model_name != "Decision Tree":
        continue
    for feature, importance in zip(feature_columns, model.feature_importances_):
        importance_rows.append({
            "Fold": fold_name,
            "Feature": feature,
            "Importance": importance,
        })
tree_importance = pd.DataFrame(importance_rows)
display(
    tree_importance.pivot(
        index="Feature", columns="Fold", values="Importance"
    ).round(3)
)

for fold_name in model_selection_folds["Fold"]:
    tree = fitted_models[(fold_name, "Decision Tree")]
    plt.figure(figsize=(18, 8))
    plot_tree(
        tree, feature_names=feature_columns,
        class_names=["No churn", "Churn"],
        filled=True, rounded=True, proportion=True, precision=2,
    )
    plt.title(f"Decision tree rules: {fold_name}")
    plt.tight_layout()
    plt.show()

## 8. Tune the decision tree

The first tree tests only one configuration, so it cannot represent the entire model family. This search varies three parameters while keeping the features, folds, purging, and validation observations fixed:

- `max_depth` controls the maximum number of successive decisions.
- `min_samples_leaf` controls the minimum number of training rows in each final leaf.
- `class_weight` tests whether giving both target classes equal importance is useful.

The primary selection criterion is mean validation PR AUC across the two folds. Minimum fold PR AUC and standard deviation are displayed to expose unstable configurations. These are model-selection results, not final unbiased performance estimates.

In [ ]:
from itertools import product

tree_search_rows = []
for max_depth, min_samples_leaf, class_weight in product(
    [2, 3, 4, 5, 6, 8, 10],
    [10, 25, 50, 100, 200],
    [None, "balanced"],
):
    for _, fold in model_selection_folds.iterrows():
        train, validation = get_fold_data(fold)
        model = DecisionTreeClassifier(
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            class_weight=class_weight,
            random_state=42,
        )
        model.fit(
            train[feature_columns].astype(float), train[target_column]
        )
        y_probability = model.predict_proba(
            validation[feature_columns].astype(float)
        )[:, 1]
        metrics = classification_metrics(
            validation[target_column], y_probability
        )
        tree_search_rows.append({
            "MaxDepth": max_depth,
            "MinSamplesLeaf": min_samples_leaf,
            "ClassWeight": str(class_weight),
            "Fold": fold["Fold"],
            **metrics,
        })

tree_search_results = pd.DataFrame(tree_search_rows)
tree_search_summary = (
    tree_search_results
    .groupby(["MaxDepth", "MinSamplesLeaf", "ClassWeight"], as_index=False)
    .agg(
        MeanPR_AUC=("PR_AUC", "mean"),
        MinimumPR_AUC=("PR_AUC", "min"),
        PR_AUCStd=("PR_AUC", "std"),
        MeanPrecision=("Precision", "mean"),
        MeanRecall=("Recall", "mean"),
    )
    .sort_values(
        ["MeanPR_AUC", "MinimumPR_AUC"], ascending=False
    )
)

tree_search_display = tree_search_summary.head(10).copy()
score_columns = [
    "MeanPR_AUC", "MinimumPR_AUC", "PR_AUCStd",
    "MeanPrecision", "MeanRecall",
]
tree_search_display[score_columns] *= 100
tree_search_display.round(2)

## 9. Evaluate the selected tree configuration

The best configuration is selected from development validation PR AUC only. Its fold-level results are compared with logistic regression. The final test remains locked, so this comparison may guide model selection but cannot yet be reported as final generalization performance.

In [ ]:
best_tree_parameters = tree_search_summary.iloc[0]
selected_class_weight = (
    None
    if best_tree_parameters["ClassWeight"] == "None"
    else best_tree_parameters["ClassWeight"]
)
print("Selected max_depth:", best_tree_parameters["MaxDepth"])
print("Selected min_samples_leaf:", best_tree_parameters["MinSamplesLeaf"])
print("Selected class_weight:", selected_class_weight)

selected_tree_rows = []
selected_trees = {}
for _, fold in model_selection_folds.iterrows():
    train, validation = get_fold_data(fold)
    model = DecisionTreeClassifier(
        max_depth=int(best_tree_parameters["MaxDepth"]),
        min_samples_leaf=int(best_tree_parameters["MinSamplesLeaf"]),
        class_weight=selected_class_weight,
        random_state=42,
    )
    model.fit(
        train[feature_columns].astype(float), train[target_column]
    )
    selected_trees[fold["Fold"]] = model
    y_probability = model.predict_proba(
        validation[feature_columns].astype(float)
    )[:, 1]
    selected_tree_rows.append({
        "Fold": fold["Fold"],
        "Model": "Tuned Decision Tree",
        **classification_metrics(validation[target_column], y_probability),
    })

selected_tree_results = pd.DataFrame(selected_tree_rows)
final_model_comparison = pd.concat([
    logistic_validation.reset_index()[selected_tree_results.columns],
    selected_tree_results,
]).sort_values(["Fold", "Model"])
final_model_comparison[percentage_columns] *= 100
final_model_comparison.round(2)

## 10. Final interpretation

The search selects `max_depth=6`, `min_samples_leaf=25`, and `class_weight='balanced'`. This tuned tree is substantially better than the initial depth-3 tree, but it still does not replace logistic regression. Validation PR AUC is lower in both periods: 89.11% versus 90.33% in fold 1 and 96.64% versus 97.10% in fold 2.

At the temporary 0.50 threshold, the tuned tree increases recall from 96.33% to 98.22% in fold 1 and from 94.91% to 98.33% in fold 2. The cost is lower precision and more false positives: 255 instead of 224 in fold 1, and 134 instead of 109 in fold 2.

This recall gain is not sufficient evidence to select the tree because the threshold has not been optimized. Since logistic regression has better PR AUC, its threshold can later be lowered to pursue higher recall while using a better underlying customer ranking. The four-feature logistic regression therefore remains the reference model. The final test remains untouched.

## 11. Random Forest, stage 1: shallow ensemble

The first Random Forest experiment isolates the effect of averaging many trees. Every tree remains shallow with `max_depth=3` and `min_samples_leaf=50`, matching the initial decision tree. We use 100 trees only to stabilize the ensemble. No class weighting or hyperparameter search is introduced at this stage.

The hypothesis is that averaging shallow trees may produce a more stable ranking than one shallow tree while preserving non-linear relationships.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

random_forest_1 = RandomForestClassifier(
    n_estimators=100,
    max_depth=3,
    min_samples_leaf=50,
    random_state=42,
    n_jobs=-1,
)

random_forest_1_rows = []
random_forest_1_fitted = {}
for _, fold in model_selection_folds.iterrows():
    train, validation = get_fold_data(fold)
    model = clone(random_forest_1)
    model.fit(
        train[feature_columns].astype(float), train[target_column]
    )
    random_forest_1_fitted[fold["Fold"]] = model

    for dataset_name, dataset in [
        ("train", train), ("validation", validation)
    ]:
        y_probability = model.predict_proba(
            dataset[feature_columns].astype(float)
        )[:, 1]
        random_forest_1_rows.append({
            "Fold": fold["Fold"],
            "Model": "Random Forest 1",
            "Dataset": dataset_name,
            **classification_metrics(dataset[target_column], y_probability),
        })

random_forest_1_results = pd.DataFrame(random_forest_1_rows)
random_forest_1_display = random_forest_1_results.copy()
random_forest_1_display[percentage_columns] *= 100
random_forest_1_display.round(2)

## 12. Compare stage 1 with logistic regression

Validation PR AUC determines whether the Random Forest improves customer ranking. Training PR AUC is shown separately to detect overfitting. Precision and recall still depend on the temporary 0.50 threshold.

In [ ]:
random_forest_1_validation = random_forest_1_results.loc[
    random_forest_1_results["Dataset"].eq("validation")
].drop(columns="Dataset")
random_forest_1_comparison = pd.concat([
    logistic_validation.reset_index()[random_forest_1_validation.columns],
    random_forest_1_validation,
]).sort_values(["Fold", "Model"])
random_forest_1_comparison[percentage_columns] *= 100
random_forest_1_comparison.round(2)


## 13. Stage 1 conclusion

The shallow Random Forest does not replace logistic regression. Its validation PR AUC is lower in both periods: 88.45% versus 90.33% in fold 1 and 96.71% versus 97.10% in fold 2.

At the temporary 0.50 threshold, recall rises to 99.58% and 99.86%, but precision falls and false positives increase from 224 to 280 in fold 1 and from 109 to 147 in fold 2. Averaging shallow trees therefore reproduces the high-recall behavior of the recency rule without improving customer ranking.

The next controlled experiment should increase only `max_depth`, while keeping `n_estimators=100`, `min_samples_leaf=50`, and `class_weight=None` unchanged. This will test whether additional interaction complexity improves PR AUC.

## 14. Random Forest, stage 2: deeper trees

This experiment increases only `max_depth` from 3 to 5. A deeper tree can represent more interactions and more precise recency or frequency thresholds. All other settings remain fixed, so any performance change can be attributed to the additional depth.

In [ ]:
random_forest_2 = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_leaf=50,
    random_state=42,
    n_jobs=-1,
)

random_forest_2_rows = []
random_forest_2_fitted = {}
for _, fold in model_selection_folds.iterrows():
    train, validation = get_fold_data(fold)
    model = clone(random_forest_2)
    model.fit(
        train[feature_columns].astype(float), train[target_column]
    )
    random_forest_2_fitted[fold["Fold"]] = model

    for dataset_name, dataset in [
        ("train", train), ("validation", validation)
    ]:
        y_probability = model.predict_proba(
            dataset[feature_columns].astype(float)
        )[:, 1]
        random_forest_2_rows.append({
            "Fold": fold["Fold"],
            "Model": "Random Forest 2",
            "Dataset": dataset_name,
            **classification_metrics(dataset[target_column], y_probability),
        })

random_forest_2_results = pd.DataFrame(random_forest_2_rows)
random_forest_2_display = random_forest_2_results.copy()
random_forest_2_display[percentage_columns] *= 100
random_forest_2_display.round(2)

## 15. Compare both complexity levels

The validation table compares logistic regression, the depth-3 forest, and the depth-5 forest. The train-validation PR AUC table checks whether the added depth creates a larger generalization gap.

In [ ]:
random_forest_2_validation = random_forest_2_results.loc[
    random_forest_2_results["Dataset"].eq("validation")
].drop(columns="Dataset")
random_forest_depth_comparison = pd.concat([
    logistic_validation.reset_index()[random_forest_2_validation.columns],
    random_forest_1_validation,
    random_forest_2_validation,
]).sort_values(["Fold", "Model"])
random_forest_depth_comparison[percentage_columns] *= 100
display(random_forest_depth_comparison.round(2))

random_forest_depth_diagnostic = pd.concat([
    random_forest_1_results, random_forest_2_results
])[["Fold", "Model", "Dataset", "PR_AUC"]]
random_forest_depth_diagnostic["PR_AUC"] *= 100
random_forest_depth_diagnostic.round(2)

## 16. Stage 2 conclusion

Increasing depth from 3 to 5 improves Random Forest validation PR AUC on both periods: from 88.45% to 89.56% in fold 1 and from 96.71% to 97.00% in fold 2. The deeper forest also recovers most of the precision lost by the shallow ensemble.

The improvement supports additional non-linear complexity, but Random Forest 2 still remains below logistic regression on both folds: 89.56% versus 90.33% and 97.00% versus 97.10%. It therefore becomes the current Random Forest reference but not the overall reference model.

The next controlled experiment can increase only `max_depth` from 5 to 7. If the gain stops or reverses, depth 5 should be retained and a different parameter investigated.

## 17. Random Forest, stage 3: depth 7

This experiment increases only `max_depth` from 5 to 7. The forest can now represent finer interactions, but the risk of fitting period-specific noise also increases. All other parameters remain fixed.

In [ ]:
random_forest_3 = RandomForestClassifier(
    n_estimators=100,
    max_depth=7,
    min_samples_leaf=50,
    random_state=42,
    n_jobs=-1,
)

random_forest_3_rows = []
for _, fold in model_selection_folds.iterrows():
    train, validation = get_fold_data(fold)
    model = clone(random_forest_3)
    model.fit(
        train[feature_columns].astype(float), train[target_column]
    )

    for dataset_name, dataset in [
        ("train", train), ("validation", validation)
    ]:
        y_probability = model.predict_proba(
            dataset[feature_columns].astype(float)
        )[:, 1]
        random_forest_3_rows.append({
            "Fold": fold["Fold"],
            "Model": "Random Forest 3",
            "Dataset": dataset_name,
            **classification_metrics(dataset[target_column], y_probability),
        })

random_forest_3_results = pd.DataFrame(random_forest_3_rows)
random_forest_3_display = random_forest_3_results.copy()
random_forest_3_display[percentage_columns] *= 100
random_forest_3_display.round(2)

## 18. Compare the three depth levels

The comparison isolates the effect of `max_depth`. A useful increase must improve validation PR AUC consistently, not only improve training performance.

In [ ]:
random_forest_3_validation = random_forest_3_results.loc[
    random_forest_3_results["Dataset"].eq("validation")
].drop(columns="Dataset")
random_forest_3_comparison = pd.concat([
    logistic_validation.reset_index()[random_forest_3_validation.columns],
    random_forest_1_validation,
    random_forest_2_validation,
    random_forest_3_validation,
]).sort_values(["Fold", "Model"])
random_forest_3_comparison[percentage_columns] *= 100
display(random_forest_3_comparison.round(2))

random_forest_3_diagnostic = pd.concat([
    random_forest_1_results,
    random_forest_2_results,
    random_forest_3_results,
])[["Fold", "Model", "Dataset", "PR_AUC"]]
random_forest_3_diagnostic["PR_AUC"] *= 100
random_forest_3_diagnostic.round(2)

## 19. Stage 3 conclusion

Increasing depth from 5 to 7 does not produce a stable improvement. Validation PR AUC changes from 89.56% to 89.59% in fold 1, a negligible gain of 0.03 percentage points, and falls from 97.00% to 96.94% in fold 2. Training PR AUC increases in both folds, which confirms that the additional capacity is used without improving temporal generalization consistently.

Depth 5 is retained for the Random Forest. It is simpler and performs at least as consistently as depth 7. The next controlled increase in complexity returns to depth 5 and reduces only `min_samples_leaf` from 50 to 40, allowing slightly more specialized customer groups while preserving the selected depth.

## 20. Random Forest, stage 4: smaller leaves

This experiment returns to the retained depth of 5 and reduces only `min_samples_leaf` from 50 to 40. Each terminal customer group may now contain fewer training observations, which adds a small amount of complexity without changing the overall tree depth.

In [ ]:
random_forest_4 = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_leaf=40,
    random_state=42,
    n_jobs=-1,
)

random_forest_4_rows = []
for _, fold in model_selection_folds.iterrows():
    train, validation = get_fold_data(fold)
    model = clone(random_forest_4)
    model.fit(
        train[feature_columns].astype(float), train[target_column]
    )

    for dataset_name, dataset in [
        ("train", train), ("validation", validation)
    ]:
        y_probability = model.predict_proba(
            dataset[feature_columns].astype(float)
        )[:, 1]
        random_forest_4_rows.append({
            "Fold": fold["Fold"],
            "Model": "Random Forest 4",
            "Dataset": dataset_name,
            **classification_metrics(dataset[target_column], y_probability),
        })

random_forest_4_results = pd.DataFrame(random_forest_4_rows)
random_forest_4_display = random_forest_4_results.copy()
random_forest_4_display[percentage_columns] *= 100
random_forest_4_display.round(2)

## 21. Compare leaf sizes 50 and 40

Only the retained depth-5 forest, the new smaller-leaf forest, and logistic regression are compared. This isolates the effect of `min_samples_leaf`.

In [ ]:
random_forest_4_validation = random_forest_4_results.loc[
    random_forest_4_results["Dataset"].eq("validation")
].drop(columns="Dataset")
random_forest_leaf_comparison = pd.concat([
    logistic_validation.reset_index()[random_forest_4_validation.columns],
    random_forest_2_validation,
    random_forest_4_validation,
]).sort_values(["Fold", "Model"])
random_forest_leaf_comparison[percentage_columns] *= 100
display(random_forest_leaf_comparison.round(2))

random_forest_leaf_diagnostic = pd.concat([
    random_forest_2_results, random_forest_4_results
])[["Fold", "Model", "Dataset", "PR_AUC"]]
random_forest_leaf_diagnostic["PR_AUC"] *= 100
random_forest_leaf_diagnostic.round(2)

## 22. Stage 4 conclusion

Reducing `min_samples_leaf` from 50 to 40 has no meaningful effect. Validation PR AUC changes from 89.56% to 89.58% in fold 1 and from 97.00% to 96.99% in fold 2. Training PR AUC changes by similarly negligible amounts.

The simpler value of 50 is retained. The higher absolute scores observed in fold 2 do not mean that the model learns from validation or necessarily learns more. All models, including logistic regression, score higher in that period, whose target prevalence and customer behavior differ from fold 1. Models must therefore be compared within each fold, and a candidate should improve both periods consistently.

The retained Random Forest remains `n_estimators=100`, `max_depth=5`, `min_samples_leaf=50`, and `class_weight=None`. Logistic regression remains the overall reference because its validation PR AUC is still higher in both folds.

## 23. IQR outlier sensitivity analysis

For each fold, IQR bounds are learned from training observations only. An observation is retained when `RecencyDays` and `PurchaseFrequency` both lie between `Q1 - 1.5 × IQR` and `Q3 + 1.5 × IQR`. The same training bounds are applied to validation.

Binary features are not filtered because an IQR of zero could incorrectly classify the minority value as an outlier. Since validation outliers are removed, these metrics describe the restricted inlier population, not all customers the business may encounter.

In [ ]:
iqr_features = ["RecencyDays", "PurchaseFrequency"]

def calculate_iqr_bounds(train):
    q1 = train[iqr_features].quantile(0.25)
    q3 = train[iqr_features].quantile(0.75)
    iqr = q3 - q1
    return pd.DataFrame({
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "LowerBound": q1 - 1.5 * iqr,
        "UpperBound": q3 + 1.5 * iqr,
    })

def get_iqr_inliers(data, bounds):
    above_lower = data[iqr_features].ge(
        bounds["LowerBound"], axis="columns"
    )
    below_upper = data[iqr_features].le(
        bounds["UpperBound"], axis="columns"
    )
    return (above_lower & below_upper).all(axis=1)

## 24. Remove outliers and retest both retained models

In [ ]:
iqr_models = {
    "IQR Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, solver="liblinear"),
    ),
    "IQR Random Forest": clone(random_forest_2),
}
iqr_model_rows = []
iqr_population_rows = []
iqr_bound_tables = []

for _, fold in model_selection_folds.iterrows():
    train, validation = get_fold_data(fold)
    bounds = calculate_iqr_bounds(train)
    train_inliers = get_iqr_inliers(train, bounds)
    validation_inliers = get_iqr_inliers(validation, bounds)
    filtered_train = train.loc[train_inliers].copy()
    filtered_validation = validation.loc[validation_inliers].copy()

    iqr_bound_tables.append(
        bounds.assign(Fold=fold["Fold"])
        .rename_axis("Feature").reset_index()
    )
    iqr_population_rows.append({
        "Fold": fold["Fold"],
        "OriginalTrainRows": len(train),
        "RemovedTrainRows": (~train_inliers).sum(),
        "RetainedTrainPercentage": train_inliers.mean(),
        "OriginalValidationRows": len(validation),
        "RemovedValidationRows": (~validation_inliers).sum(),
        "RetainedValidationPercentage": validation_inliers.mean(),
        "AffectedValidationCustomers": validation.loc[
            ~validation_inliers, "Customer ID"
        ].nunique(),
    })

    for model_name, model_template in iqr_models.items():
        model = clone(model_template)
        model.fit(
            filtered_train[feature_columns].astype(float),
            filtered_train[target_column],
        )
        for dataset_name, dataset in [
            ("train", filtered_train),
            ("validation", filtered_validation),
        ]:
            probability = model.predict_proba(
                dataset[feature_columns].astype(float)
            )[:, 1]
            iqr_model_rows.append({
                "Fold": fold["Fold"],
                "Model": model_name,
                "Dataset": dataset_name,
                "Rows": len(dataset),
                **classification_metrics(
                    dataset[target_column], probability
                ),
            })

iqr_bounds = pd.concat(iqr_bound_tables, ignore_index=True)
iqr_population = pd.DataFrame(iqr_population_rows)
iqr_results = pd.DataFrame(iqr_model_rows)
iqr_population_display = iqr_population.copy()
iqr_population_display[[
    "RetainedTrainPercentage", "RetainedValidationPercentage"
]] *= 100
display(iqr_bounds.round(2))
display(iqr_population_display.round(2))

iqr_validation_display = iqr_results.loc[
    iqr_results["Dataset"].eq("validation")
].copy()
iqr_validation_display[percentage_columns] *= 100
iqr_validation_display.round(2)

## 25. IQR experiment conclusion

The IQR rule finds no `RecencyDays` outlier because the learned upper bounds exceed the maximum eligible recency. All removed observations have `PurchaseFrequency > 11`. The filter removes 338 validation rows in fold 1 and 554 in fold 2, representing 8.82% and 17.58% of the two validation populations.

PR AUC rises slightly after filtering, but the evaluation population has also changed. The excluded observations represent highly frequent customers who may be commercially valuable and are valid rather than erroneous. Better metrics on this restricted population do not prove better generalization to the complete customer population.

IQR filtering is therefore retained as a sensitivity analysis but is not adopted in the default pipeline. A deployment model must score high-frequency customers rather than silently exclude them. Robust transformations or a dedicated high-frequency segment would be safer alternatives.

## 26. Stochastic Gradient Descent, stage 1

`SGDClassifier` with logistic loss learns a linear probability boundary similar to logistic regression, but updates its coefficients incrementally. It is useful for large or streaming datasets. This first experiment uses L2 regularization, the complete customer population, and no class weighting. Scaling is learned inside each training fold.

In [ ]:
from sklearn.linear_model import SGDClassifier

sgd_1 = make_pipeline(
    StandardScaler(),
    SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=0.0001,
        max_iter=1000,
        tol=0.001,
        random_state=42,
    ),
)

sgd_1_rows = []
sgd_1_fitted = {}
for _, fold in model_selection_folds.iterrows():
    train, validation = get_fold_data(fold)
    model = clone(sgd_1)
    model.fit(
        train[feature_columns].astype(float), train[target_column]
    )
    sgd_1_fitted[fold["Fold"]] = model
    for dataset_name, dataset in [
        ("train", train), ("validation", validation)
    ]:
        y_probability = model.predict_proba(
            dataset[feature_columns].astype(float)
        )[:, 1]
        sgd_1_rows.append({
            "Fold": fold["Fold"],
            "Model": "SGD 1",
            "Dataset": dataset_name,
            **classification_metrics(dataset[target_column], y_probability),
        })

sgd_1_results = pd.DataFrame(sgd_1_rows)
sgd_1_display = sgd_1_results.copy()
sgd_1_display[percentage_columns] *= 100
sgd_1_display.round(2)

## 27. SGD stage 1 comparison

The first SGD model is compared with logistic regression and the retained Random Forest on the same full validation observations.

In [ ]:
sgd_1_validation = sgd_1_results.loc[
    sgd_1_results["Dataset"].eq("validation")
].drop(columns="Dataset")
sgd_model_comparison = pd.concat([
    logistic_validation.reset_index()[sgd_1_validation.columns],
    random_forest_2_validation,
    sgd_1_validation,
]).sort_values(["Fold", "Model"])
sgd_model_comparison[percentage_columns] *= 100
sgd_model_comparison.round(2)

## 28. SGD stage 1 conclusion

SGD produces almost the same ranking as logistic regression, which is expected because both models learn a linear logistic boundary. Its PR AUC is slightly lower in fold 1, 90.14% versus 90.33%, and slightly higher in fold 2, 97.12% versus 97.10%.

These differences are negligible and not consistent across time. SGD also misses more churners in fold 2 at the temporary 0.50 threshold. Logistic regression remains the reference because it is equally effective, easier to fit reproducibly on this dataset, and does not require an incremental-learning capability.

## 29. Error analysis of the reference model

This section profiles true positives, false positives, false negatives, and true negatives from the retained logistic regression. Feature differences are associations with model errors, not proof that a feature causes an error. The analysis uses the full validation population at the temporary 0.50 threshold.

In [ ]:
import numpy as np

error_frames = []
for _, fold in model_selection_folds.iterrows():
    _, validation = get_fold_data(fold)
    model = fitted_models[(fold["Fold"], "Logistic Regression")]
    y_probability = model.predict_proba(
        validation[feature_columns].astype(float)
    )[:, 1]
    analysis = validation[[
        "Customer ID", "ReferenceDate", *feature_columns, target_column
    ]].copy()
    analysis["PredictedProbability"] = y_probability
    analysis["PredictedChurn"] = y_probability >= 0.50
    analysis["ErrorType"] = np.select(
        [
            analysis[target_column] & analysis["PredictedChurn"],
            ~analysis[target_column] & analysis["PredictedChurn"],
            analysis[target_column] & ~analysis["PredictedChurn"],
        ],
        ["True positive", "False positive", "False negative"],
        default="True negative",
    )
    analysis["Fold"] = fold["Fold"]
    error_frames.append(analysis)

error_analysis = pd.concat(error_frames, ignore_index=True)
error_counts = error_analysis.pivot_table(
    index="Fold", columns="ErrorType",
    values="Customer ID", aggfunc="size", fill_value=0,
)
error_feature_profiles = (
    error_analysis.groupby(["Fold", "ErrorType"])
    .agg(
        Snapshots=("Customer ID", "size"),
        MedianRecencyDays=("RecencyDays", "median"),
        MedianPurchaseFrequency=("PurchaseFrequency", "median"),
        CancellationRate=("HasCancellation", "mean"),
        RiskWindowRate=("IsInChurnRiskWindow", "mean"),
        MeanPredictedProbability=("PredictedProbability", "mean"),
    )
    .reset_index()
)
profile_percentages = [
    "CancellationRate", "RiskWindowRate", "MeanPredictedProbability"
]
error_feature_profiles[profile_percentages] *= 100
display(error_counts)
error_feature_profiles.round(2)

## 30. Error-profile interpretation

False negatives have lower median recency than true positives but much higher purchase frequency. Their median frequencies are 9 and 12 across the two folds, compared with 3 for true positives. Approximately 94% to 96% of false negatives also have a cancellation history. The model interprets these historically engaged customers as likely to return, but some still reach the churn deadline.

False positives are all inside the churn-risk window and have median recencies of approximately 39 and 42 days. They appear close to churn at the reference date but purchase before reaching 60 days of inactivity. The explicit risk-window indicator cannot separate them from true positives because both groups almost always have the same value of 1.

The difficult region is therefore not the obvious low-risk population. It is the group already inside the risk window, especially customers whose high historical frequency or cancellation behavior conflicts with their current inactivity. These associations motivate threshold and segment analysis rather than removing the observations.

## 31. Feature signals across the retained models

Linear-model coefficients are comparable within each fold because the features are standardized. Their sign gives the direction of the association with predicted churn. Tree and Random Forest impurity importance is always non-negative and sums to one; it shows how often a feature improves splits, not the direction or a causal effect. Values from these two importance families must not be compared numerically with one another.

In [ ]:
linear_signal_rows = []
tree_importance_rows = []
for fold_name in model_selection_folds["Fold"]:
    for model_name, model in [
        ("Logistic Regression", fitted_models[(fold_name, "Logistic Regression")]),
        ("SGD", sgd_1_fitted[fold_name]),
    ]:
        coefficients = model[-1].coef_[0]
        for feature, coefficient in zip(feature_columns, coefficients):
            linear_signal_rows.append({
                "Fold": fold_name,
                "Model": model_name,
                "Feature": feature,
                "StandardizedCoefficient": coefficient,
            })

    for model_name, model in [
        ("Tuned Decision Tree", selected_trees[fold_name]),
        ("Random Forest", random_forest_2_fitted[fold_name]),
    ]:
        for feature, importance in zip(feature_columns, model.feature_importances_):
            tree_importance_rows.append({
                "Fold": fold_name,
                "Model": model_name,
                "Feature": feature,
                "ImpurityImportance": importance,
            })

linear_feature_signals = pd.DataFrame(linear_signal_rows)
tree_feature_signals = pd.DataFrame(tree_importance_rows)
display(
    linear_feature_signals.pivot_table(
        index="Feature", columns=["Model", "Fold"],
        values="StandardizedCoefficient",
    ).round(3)
)
tree_feature_signals.pivot_table(
    index="Feature", columns=["Model", "Fold"],
    values="ImpurityImportance",
).round(3)

## 32. Feature-signal interpretation

The linear models identify the same stable directions in both folds. `IsInChurnRiskWindow` has the strongest positive coefficient, followed by `RecencyDays`: current inactivity increases predicted churn risk. `PurchaseFrequency` has a strong negative coefficient, meaning historically frequent customers are considered more likely to purchase before their churn deadline. `HasCancellation` has a smaller negative coefficient after the other variables are controlled. This may represent prior engagement rather than a protective causal effect.

The Random Forest assigns approximately 57% of impurity importance to `RecencyDays`, 40% to `IsInChurnRiskWindow`, 3% to `PurchaseFrequency`, and less than 1% to `HasCancellation`. The tuned decision tree switches almost all importance between recency and the risk-window indicator across folds. This instability occurs because `IsInChurnRiskWindow` is directly derived from recency, so trees may select either correlated representation.

The feature results therefore point to one dominant inactivity signal, one useful historical-frequency signal, and a weak cancellation signal. The two recency variables should be treated as related representations rather than two independent business drivers.

## 33. Business cost function

Model-ranking metrics do not determine the campaign threshold. A false positive creates an unnecessary intervention, while a false negative leaves a future churner untreated. The simplified objective is:

`TotalCost = CostFalsePositive × FalsePositives + CostFalseNegative × FalseNegatives`

Because actual monetary costs are unavailable, `CostFalsePositive` is fixed to one unit and `CostFalseNegative` is tested at 1, 2, 5, and 10 units. Threshold selection uses development validation only.

In [ ]:
def calculate_business_cost(
    y_true, y_probability, threshold,
    false_positive_cost, false_negative_cost,
):
    y_pred = y_probability >= threshold
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[False, True]
    ).ravel()
    return {
        "Threshold": threshold,
        "TotalCost": false_positive_cost * fp + false_negative_cost * fn,
        "FalsePositives": fp,
        "FalseNegatives": fn,
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "ContactRate": y_pred.mean(),
    }

cost_rows = []
candidate_thresholds = np.arange(0.01, 1.00, 0.01)
false_negative_costs = [1, 2, 5, 10]

for _, fold in model_selection_folds.iterrows():
    _, validation = get_fold_data(fold)
    fold_name = fold["Fold"]
    cost_models = {
        "Logistic Regression": fitted_models[(fold_name, "Logistic Regression")],
        "Tuned Decision Tree": selected_trees[fold_name],
        "Random Forest": random_forest_2_fitted[fold_name],
        "SGD": sgd_1_fitted[fold_name],
    }
    for model_name, model in cost_models.items():
        y_probability = model.predict_proba(
            validation[feature_columns].astype(float)
        )[:, 1]
        for false_negative_cost in false_negative_costs:
            threshold_costs = [
                calculate_business_cost(
                    validation[target_column], y_probability, threshold,
                    false_positive_cost=1,
                    false_negative_cost=false_negative_cost,
                )
                for threshold in candidate_thresholds
            ]
            best_cost = min(
                threshold_costs,
                key=lambda result: (result["TotalCost"], result["ContactRate"]),
            )
            cost_rows.append({
                "Fold": fold_name,
                "Model": model_name,
                "FalsePositiveCost": 1,
                "FalseNegativeCost": false_negative_cost,
                "CostRatio_FN_to_FP": false_negative_cost,
                **best_cost,
            })

business_cost_results = pd.DataFrame(cost_rows)
business_cost_results["CostPer1000Snapshots"] = (
    business_cost_results["TotalCost"]
    / business_cost_results["Fold"].map(
        error_analysis.groupby("Fold").size()
    ) * 1000
)
cost_display = business_cost_results.copy()
cost_display[["Precision", "Recall", "ContactRate"]] *= 100
cost_display.sort_values(
    ["CostRatio_FN_to_FP", "Fold", "TotalCost"]
).round(2)

## 34. Cost-analysis interpretation

With equal false-positive and false-negative costs, logistic regression has the lowest total cost on both folds. When one false negative costs at least five times one false positive, the retained Random Forest becomes the lowest-cost model on both periods because it misses very few churners.

At a cost ratio of two, the models are nearly tied and the preferred model varies by period. The selected thresholds also differ substantially between folds, which indicates that a single operational threshold should not yet be frozen. Probability calibration and business constraints must be addressed next.

These cost units are hypothetical. The final objective should include campaign cost, customer value, retained margin, intervention success probability, and the cost of contacting true positives as well as false positives.

## 35. Consolidated validation results

This final table gathers every trained model tested on the complete validation population. IQR-filtered results remain separate because they use a different population and are not directly comparable.

In [ ]:
summary_columns = [
    "Fold", "Model", "Precision", "Recall", "PR_AUC",
    "FalsePositives", "FalseNegatives",
]
all_model_validation_results = pd.concat([
    logistic_validation.reset_index()[summary_columns],
    tree_validation.reset_index()[summary_columns],
    selected_tree_results[summary_columns],
    random_forest_1_validation[summary_columns],
    random_forest_2_validation[summary_columns],
    random_forest_3_validation[summary_columns],
    random_forest_4_validation[summary_columns],
    sgd_1_validation[summary_columns],
]).sort_values(["Fold", "PR_AUC"], ascending=[True, False])
all_model_validation_results[["Precision", "Recall", "PR_AUC"]] *= 100
display(all_model_validation_results.round(2))

iqr_validation_summary = iqr_results.loc[
    iqr_results["Dataset"].eq("validation"), summary_columns
].copy()
iqr_validation_summary[["Precision", "Recall", "PR_AUC"]] *= 100
iqr_validation_summary.round(2)

## 36. Model-selection summary

Logistic regression remains the reference model. It has the highest PR AUC in fold 1 and is effectively tied with SGD in fold 2, while remaining simpler and more stable than the alternatives. Random Forest provides higher recall at some thresholds and becomes economically preferable in the illustrative scenarios where a false negative costs at least five times a false positive.

IQR filtering is not adopted because it removes valid high-frequency customers and changes the evaluation population. Error analysis shows that most difficult cases are already inside the churn-risk window, where frequency and cancellation history can conflict with current inactivity. Recency and the 30-day risk-window indicator dominate prediction, purchase frequency adds a stable negative signal, and cancellation history contributes only a weak conditional signal.